# CorroborAI — démonstration

Ce notebook déroule le scénario de démonstration demandé par les consignes, sur les fichiers du défi :

1. **un cas conforme**, **un écart justifié automatiquement** et **une vraie anomalie**, chacun avec sa justification et les données qui l'ont produit ;
2. la vue de l'investigateur : écarts triés par priorité, motifs et règles candidates ;
3. la **rétroaction experte** : une règle prévisualisée puis appliquée ;
4. la **couche IA** encadrée et la protection des données ;
5. le **rapport** exportable.

**Prérequis :** `pip install -e .` à la racine du dépôt, et les fichiers du défi dans `data/` (ou le chemin indiqué par `CORROBORAI_DATA_DIR`).

In [1]:
import os
from pathlib import Path

import pandas as pd

from corroborai.engine import corroborate
from corroborai.io.loaders import load_bundle
from corroborai.rules_config import load_rules
from corroborai.service import evidence_rows, findings_frame, investigate

pd.set_option("display.max_colwidth", 140)

# Répertoire des données : CORROBORAI_DATA_DIR, sinon data/ à la racine du dépôt
racine = Path.cwd() if (Path.cwd() / "config").is_dir() else Path.cwd().parent
DATA = Path(os.environ.get("CORROBORAI_DATA_DIR", racine / "data"))
print("Données :", DATA)

Données : /mnt/project


## 1. Chargement en lecture seule

Chaque fichier est lu une seule fois en octets ; son empreinte SHA-256 est calculée sur exactement les octets analysés. Si un `manifest.json` est présent, les empreintes y sont comparées à titre informatif.

In [2]:
bundle = load_bundle(DATA)

pd.DataFrame(bundle.integrity.to_records())[["fichier_logique", "fichier_lu", "requis", "statut", "note"]]

,fichier_logique,fichier_lu,requis,statut,note
0,source,Employe_Source_Anonymise_VF.xlsx,True,CONFORME_AU_MANIFESTE,
1,target,Employe_Destination_Anonymise_VF.xlsx,True,CONFORME_AU_MANIFESTE,
2,poste_detail,détail_du_poste.xlsx,True,CONFORME_AU_MANIFESTE,
3,motifs,Motif_de_la_situation_demploi.xlsx,True,CONFORME_AU_MANIFESTE,trouvé sous le nom « Motif_de_la_situation_demploi.xlsx »
4,mapping,Mapping.xlsx,False,CONFORME_AU_MANIFESTE,


In [3]:
pd.DataFrame(
    [(t.name, t.path.name, len(t.df)) for t in (bundle.source, bundle.target, bundle.poste_detail, bundle.motifs)],
    columns=["table", "fichier", "lignes"],
)

,table,fichier,lignes
0,source,Employe_Source_Anonymise_VF.xlsx,23
1,target,Employe_Destination_Anonymise_VF.xlsx,22
2,poste_detail,détail_du_poste.xlsx,114
3,motifs,Motif_de_la_situation_demploi.xlsx,92


## 2. Corroboration

Appariement des affectations, application des 25 règles du mapping codifié, réévaluation sous les interprétations alternatives, moteur d'hypothèses et score de priorité. Les verdicts sont fixés par les règles ; l'analyse les explique sans les modifier.

In [4]:
cfg = load_rules()
result = corroborate(bundle, cfg)

print(f"{len(result.findings)} verdicts — {sum(p.matched for p in result.pairs)} affectations appariées sur {len(result.pairs)}")
print("Fichiers sources après traitement :", "inchangés" if result.integrity_after.ok else "MODIFIÉS")
pd.Series(result.counts(), name="verdicts").to_frame()

551 verdicts — 22 affectations appariées sur 23
Fichiers sources après traitement : inchangés


,verdicts
CONFORME,361
JUSTIFIE,128
ANOMALIE,62
INDETERMINE,0


Une petite fonction pour afficher la fiche complète d'un écart : valeurs comparées, verdict, justification, règle et référence du mapping, cause probable, priorité, puis **les lignes d'origine ayant servi à la décision**.

In [5]:
FICHE = ["verdict", "subcategory", "source_raw", "expected", "target_raw", "justification", "rule_id", "rule_ref",
         "confidence", "rule_params", "probable_cause", "priority_breakdown"]


def fiche(finding_id):
    f = next(x for x in result.findings if x.finding_id == finding_id)
    rec = f.to_record()
    display(pd.DataFrame({"valeur": {k: rec[k] for k in FICHE if rec[k] not in ("", None)}}))
    for ev, row in evidence_rows(bundle, f):
        valeurs = {c: row[c] for c in ev.values if c in row} or ev.values
        print(f"— preuve {ev.id} : table {ev.table}, ligne {ev.row}" + (f" ({ev.note})" if ev.note else ""))
        print("  ", valeurs)
    return f

## 3. Trois verdicts

### Cas 1 — conforme : date de début d'affectation de l'employé 2173396

Le poste 69289 a changé trois fois d'unité administrative (325 → 320 → 352). La règle retient l'intersection des intervalles : début = MAX(date d'entrée, début de l'unité courante). Elle retrouve exactement la date de la cible. La lecture littérale de la règle (« date la plus ancienne ») aurait donné une autre date : le verdict en dépend, donc la confiance est `MOYENNE` et le verdict alternatif est tracé.

In [6]:
_ = fiche("2173396:P:69289:assignmentStartDate")

,valeur
verdict,CONFORME
source_raw,2019-04-29 00:00:00
expected,2019-04-29
target_raw,2019-04-29 00:00:00
justification,Valeur identique à la source après normalisation de format : 2019-04-29.
rule_id,R-ASSIGN-START
rule_ref,"Mapping!46, Mapping!48, Jointure - Détail du poste!2"
confidence,MOYENNE
rule_params,"alternative[strict_literal]=ANOMALIE (attendu 1997-06-24), interpretation=INT-ASSIGN-DATES=interval_intersection"


— preuve src:15 : table source, ligne 15
   {'DateEntréePoste': '2019-04-29 00:00:00', 'CodePoste': '69289', 'CodeDirection': '352'}
— preuve tgt:14 : table target, ligne 14
   {'assignmentStartDate': '2019-04-29 00:00:00'}
— preuve poste:69 : table poste_detail, ligne 69 (Historique du détail du poste)
   {'date_effet': '1987-11-21', 'unite_admin': '325'}
— preuve poste:70 : table poste_detail, ligne 70 (Historique du détail du poste)
   {'date_effet': '1991-06-18', 'unite_admin': '320'}
— preuve poste:71 : table poste_detail, ligne 71 (Historique du détail du poste)
   {'date_effet': '1993-12-14', 'unite_admin': '320'}
— preuve poste:72 : table poste_detail, ligne 72 (Historique du détail du poste)
   {'date_effet': '1997-06-24', 'unite_admin': '352'}


### Cas 2 — écart justifié automatiquement : motif d'absence de l'employé 2911996

La source porte le motif `807`, la cible `170`. Une comparaison brute dirait « différent ». La règle de situation d'emploi s'applique : l'employé est en absence complète (code de traitement des accès 02), donc le code attendu est le code Remphor du motif, obtenu par jointure avec la table des motifs (ligne 33).

In [7]:
_ = fiche("2911996:P:12548:statusReasonCode")

,valeur
verdict,JUSTIFIE
source_raw,807
expected,170
target_raw,170
justification,valeur cible « 170 » conforme à la règle : situation « Absence complète » : code Remphor du motif « 807 ».
rule_id,R-SITUATION-CAD
rule_ref,"Mapping!17, Mapping!19, Règles situation d'emploi!3, Jointure - Motif des situations!2"
confidence,ELEVEE
rule_params,"alternative[code_statut_emploi]=JUSTIFIE (attendu 170), alternative[motif_code_gestion_acces]=JUSTIFIE (attendu 170), interpretation=INT..."


— preuve src:16 : table source, ligne 16
   {'CodeSuspensionAccès': '2', 'CodeRaisonStatut': '807', 'CodeStatutEmploi': '2'}
— preuve tgt:15 : table target, ligne 15
   {'statusReasonCode': '170'}
— preuve motif:33 : table motifs, ligne 33 (Motif de la situation d'emploi)
   {'CodeCatégorieStatut': '807', 'CodeStatutSystèmeExterne': '170', 'CodeGestionAccès': '2'}


### Cas 3 — vraie anomalie : type d'employé de l'employé 2762457

L'employé est permanent à temps plein : la règle attend `JWN`, la cible contient `WHX`. Le moteur d'hypothèses constate que l'employé 4625374 présente l'écart inverse : il s'agit probablement d'une **permutation** entre les deux enregistrements. Erreur isolée et certaine, avec sa contrepartie identifiée : priorité maximale.

In [8]:
_ = fiche("2762457:P:30106:contractTypeCode")

,valeur
verdict,ANOMALIE
source_raw,V / Oui / Oui
expected,JWN
target_raw,WHX
justification,"Valeur attendue « JWN » selon la règle (CatégorieEmploi=V et EstPermanent=Oui et EstTempsPlein=Oui → « JWN » (Permanent temps plein)), v..."
rule_id,R-CONTRACT
rule_ref,"Mapping!24, Mapping!26, Mapping!39, Mapping!40"
confidence,ELEVEE
rule_params,"hypothèses testées=H-PERMUTATION, H-HISTORY-RECORD, H-ALT-SOURCE, H-ALT-INTERPRETATION, H-BIJECTION, H-SYSTEMIC, H-TYPE-ABSENT, interpre..."
probable_cause,Permutation entre deux employés : Valeur cible « WHX » = valeur attendue de l'employé 4625374 ; sa valeur cible « JWN » = la valeur atte...


— preuve src:8 : table source, ligne 8
   {'CatégorieEmploi': 'V', 'EstPermanent': 'Oui', 'EstTempsPlein': 'Oui'}
— preuve tgt:7 : table target, ligne 7
   {'contractTypeCode': 'WHX'}
— preuve src:9 : table source, ligne 9 (Contrepartie : employé 4625374)
   {'CatégorieEmploi': 'O', 'EstPermanent': 'Non', 'EstTempsPlein': 'Non'}
— preuve tgt:8 : table target, ligne 8 (Contrepartie : employé 4625374)
   {'contractTypeCode': 'JWN'}


## 4. La vue de l'investigateur

Les écarts à investiguer, triés par priorité. Les erreurs isolées (affectation absente, permutations) passent devant les artefacts systémiques (courriels et libellés d'emploi), qui se corrigent une seule fois.

In [9]:
inv = findings_frame(investigate(result.findings))
inv[["Priorité", "Employé", "Champ", "Verdict", "Cause probable"]].head(20)

,Priorité,Employé,Champ,Verdict,Cause probable
0,100.0,1545850,affectation,ANOMALIE,"Type d'affectation absent du système cible : Aucune affectation de type A (Temporaire) dans le système cible (0 sur 22), contre 1 dans l..."
1,100.0,2762457,contractTypeCode,ANOMALIE,Permutation entre deux employés : Valeur cible « WHX » = valeur attendue de l'employé 4625374 ; sa valeur cible « JWN » = la valeur atte...
2,100.0,3712987,contractTypeCode,ANOMALIE,Permutation entre deux employés : Valeur cible « XFLR » = valeur attendue de l'employé 7254364 ; sa valeur cible « JWN » = la valeur att...
3,100.0,4625374,contractTypeCode,ANOMALIE,Permutation entre deux employés : Valeur cible « JWN » = valeur attendue de l'employé 2762457 ; sa valeur cible « WHX » = la valeur atte...
4,100.0,7254364,contractTypeCode,ANOMALIE,Permutation entre deux employés : Valeur cible « JWN » = valeur attendue de l'employé 3712987 ; sa valeur cible « XFLR » = la valeur att...
5,80.0,3241002,assignmentStartDate,ANOMALIE,Valeur d'un autre enregistrement de l'historique du poste : Date cible 2022-10-05 = date d'effet la plus récente de l'historique du post...
6,80.0,4402456,assignmentStartDate,ANOMALIE,Valeur d'un autre enregistrement de l'historique du poste : Date cible 2013-10-20 = date d'effet la plus récente de l'historique du post...
7,80.0,9989151,assignmentStartDate,ANOMALIE,Valeur d'un autre enregistrement de l'historique du poste : Date cible 2021-03-30 = date d'effet la plus récente de l'historique du post...
8,70.0,2911996,dailyHoursOverride,ANOMALIE,Valeur provenant d'une autre colonne : Valeur cible « 8 » = détail du poste (enregistrement le plus récent).HeuresJourContrat ; cette co...
9,70.0,2911996,weeklyHoursOverride,ANOMALIE,Valeur provenant d'une autre colonne : Valeur cible « 40 » = détail du poste (enregistrement le plus récent).HeuresSemaineContrat ; cett...


### Motifs et règles candidates

In [10]:
pd.DataFrame([(p.hypothesis_id, p.target_field, p.count, p.description) for p in result.analysis.patterns],
             columns=["hypothèse", "champ", "verdicts", "description"])

,hypothèse,champ,verdicts,description
0,H-HISTORY-RECORD,assignmentStartDate,3,"Motif récurrent : pour 3 anomalies, la cible contient la date d'effet la plus récente du détail du poste au lieu de la date calculée par..."
1,H-SYSTEMIC,contactEmail,22,"22 enregistrements sur 22 (100%) sont en écart pour ce champ (type dominant : identifiant incohérent avec le matricule, 22/22)."
2,H-PERMUTATION,contractTypeCode,2,Valeurs inversées entre les employés 2762457 et 4625374 (« JWN » ↔ « WHX »).
3,H-PERMUTATION,contractTypeCode,2,Valeurs inversées entre les employés 3712987 et 7254364 (« JWN » ↔ « XFLR »).
4,H-ALT-SOURCE,dailyHoursOverride,4,« dailyHoursOverride » semble alimenté par détail du poste (enregistrement le plus récent).HeuresJourContrat (concordance 100% sur 22 en...
5,H-SYSTEMIC,positionName,22,"22 enregistrements sur 22 (100%) sont en écart pour ce champ (type dominant : code et libellé différents, 22/22)."
6,H-BIJECTION,positionName,22,Correspondance un-pour-un stable sur 22 écarts : 9 valeurs attendues distinctes recodées de façon constante (ex. « 6031-Empl6031 » → « 0...
7,H-PERMUTATION,siteName,2,Valeurs inversées entre les employés 3241002 et 6035643 (« Emplacement48 » ↔ « Emplacement35 »).
8,H-ALT-SOURCE,weeklyHoursOverride,4,« weeklyHoursOverride » semble alimenté par détail du poste (enregistrement le plus récent).HeuresSemaineContrat (concordance 100% sur 2...
9,H-TYPE-ABSENT,affectation,1,"Aucune affectation de type A (Temporaire) dans le système cible (0 sur 22), contre 1 dans la source."


In [11]:
pd.DataFrame([(c.target_field, c.column, f"{c.support:.0%}", f"{c.mapped_support:.0%}", len(c.explains))
              for c in result.analysis.candidate_rules],
             columns=["champ", "colonne candidate", "concordance", "règle actuelle", "anomalies expliquées"])

,champ,colonne candidate,concordance,règle actuelle,anomalies expliquées
0,dailyHoursOverride,détail du poste (enregistrement le plus récent).HeuresJourContrat,100%,82%,4
1,weeklyHoursOverride,détail du poste (enregistrement le plus récent).HeuresSemaineContrat,100%,82%,4


### Interprétations : taux d'accord avec la cible

Pour la date de début d'affectation, l'intersection d'intervalles s'accorde avec la cible sur 86 % des enregistrements ; la lecture littérale, sur aucun.

In [12]:
pd.DataFrame([(s.interpretation, s.target_field, s.choice, "oui" if s.active else "", f"{s.rate:.0%}")
              for s in result.interpretation_stats if s.target_field in ("assignmentStartDate", "divisionName")],
             columns=["interprétation", "champ", "choix", "retenu", "accord avec la cible"])

,interprétation,champ,choix,retenu,accord avec la cible
0,INT-ASSIGN-DATES,assignmentStartDate,interval_intersection,oui,86%
1,INT-ASSIGN-DATES,assignmentStartDate,strict_literal,,0%
2,INT-CONCAT-PADDING,divisionName,zero_padded,oui,100%
3,INT-CONCAT-PADDING,divisionName,unpadded,,0%


## 5. Rétroaction experte

Une règle candidate devient une **règle experte** seulement après un aperçu exécuté à blanc, qui montre exactement quels verdicts changeraient, puis l'acceptation de l'expert.

In [13]:
from corroborai.feedback import ExpertRule, FeedbackStore, preview_rule

candidate = next(c for c in result.analysis.candidate_rules if c.target_field == "weeklyHoursOverride")
regle = ExpertRule("", "accept_alternative_source", candidate.target_field, {"column": candidate.column},
                   "Les heures du système Temps proviennent du contrat du poste.")
apercu = preview_rule(regle, result.findings, result.rule_context, set(cfg.targets))
print(apercu.summary())
findings_frame(apercu.matched)[["Employé", "Champ", "Valeur attendue", "Valeur cible"]]

4 anomalie(s) sur 4 pour « weeklyHoursOverride » deviendraient JUSTIFIE ; 0 resteraient à investiguer.


,Employé,Champ,Valeur attendue,Valeur cible
0,2911996,weeklyHoursOverride,35,40
1,3712987,weeklyHoursOverride,,40
2,4402456,weeklyHoursOverride,36,40
3,7683990,weeklyHoursOverride,36,40


Le fichier d'exemple `feedback/exemple_retroaction.yaml` contient trois règles et une correction. Appliqué à la corroboration :

In [14]:
exemple = FeedbackStore.load(racine / "feedback" / "exemple_retroaction.yaml", set(cfg.targets))
avec_retroaction = corroborate(bundle, cfg, feedback=exemple)

print("Règles appliquées :", {k: len(v) for k, v in avec_retroaction.feedback.rules_applied.items()})
print("Corrections appliquées :", avec_retroaction.feedback.corrections_applied)
pd.DataFrame({"sans rétroaction": result.counts(), "avec rétroaction": avec_retroaction.counts()})

Règles appliquées : {'R-EXPERT-001': 4, 'R-EXPERT-002': 4, 'R-EXPERT-003': 22}
Corrections appliquées : ['1545850:P:35342:contactEmail']


,sans rétroaction,avec rétroaction
CONFORME,361,361
JUSTIFIE,128,159
ANOMALIE,62,31
INDETERMINE,0,0


## 6. Couche IA encadrée

Par défaut, le fournisseur est le **gabarit** : aucun LLM, résultat reproductible et hors ligne. Avec Gemini ou un modèle local, seuls les textes changent ; les verdicts et les priorités restent identiques.

In [15]:
from corroborai.ai.tasks import run_ai

rapport_ia = run_ai(result, bundle)                       # fournisseur par défaut : gabarit
# rapport_ia = run_ai(result, bundle, provider_name="gemini")   # nécessite GEMINI_API_KEY
# rapport_ia = run_ai(result, bundle, provider_name="local")    # serveur compatible OpenAI local

print("Fournisseur :", rapport_ia.provider, "| tâches :", dict(rapport_ia.statuses))
print()
print(next(s.synthese for s in rapport_ia.summaries if s.scope == "GLOBALE"))

Fournisseur : gabarit | tâches : {'GABARIT': 12}

62 anomalie(s) et 0 cas indéterminé(s). 12 motif(s) ou règle(s) candidate(s) identifié(s). Motif récurrent : pour 3 anomalies, la cible contient la date d'effet la plus récente du détail du poste au lieu de la date calculée par la règle. 22 enregistrements sur 22 (100%) sont en écart pour ce champ (type dominant : identifiant incohérent avec le matricule, 22/22). Valeurs inversées entre les employés 2762457 et 4625374 (« JWN » ↔ « WHX »).


### Protection des données

Avant tout envoi à un fournisseur externe, le dossier est pseudonymisé et la politique de flux est évaluée. Voici un extrait du dossier qui serait envoyé pour l'employé 2762457 : les matricules et courriels n'y figurent pas.

In [16]:
from corroborai.ai.policy import Pseudonymizer, check_flow, load_llm_config
from corroborai.ai.tasks import employee_dossier

pseudo = Pseudonymizer.from_bundle(bundle)
ecarts = [f for f in result.findings if f.person_id == "2762457" and f.priority is not None]
dossier = employee_dossier("2762457", ecarts, pseudo, exclude_systemic=False)
llm = load_llm_config()

print("Décision de flux vers Gemini :", check_flow(dossier, llm.provider("gemini"), llm.flow, pseudo).summary())
print("Matricule présent dans la charge :", "2762457" in dossier.serialize())
print()
print(dossier.serialize()[:900], "…")

Décision de flux vers Gemini : AUTORISÉ
Matricule présent dans la charge : False

{
 "ecarts": [
  {
   "cause_probable": "Permutation entre deux employés : Valeur cible « WHX » = valeur attendue de l'employé EMP-10 ; sa valeur cible « JWN » = la valeur attendue ici.",
   "champ": "contractTypeCode",
   "hypotheses": [
    "H-PERMUTATION"
   ],
   "justification": "Valeur attendue « JWN » selon la règle (CatégorieEmploi=V et EstPermanent=Oui et EstTempsPlein=Oui → « JWN » (Permanent temps plein)), valeur cible « WHX ».",
   "preuves": [
    "src:8",
    "tgt:7",
    "src:9",
    "tgt:8"
   ],
   "priorite": 100.0,
   "ref": "E1",
   "regle": "R-CONTRACT",
   "sous_categorie": "",
   "valeur_attendue": "JWN",
   "valeur_cible": "WHX",
   "verdict": "ANOMALIE"
  },
  {
   "cause_probable": "Écart systémique : 22 enregistrements sur 22 (100%) sont en écart pour ce champ (type dominant : identifiant incohérent avec le matricule, 22/22).",
   "champ": "contactEmail",
   "h …


## 7. Rapport exportable

In [17]:
from corroborai.report import write_csv, write_report

sortie = racine / "out"
xlsx = write_report(result, sortie / "rapport_corroboration.xlsx")
csv = write_csv(result, sortie / "verdicts.csv")
print("Rapport :", xlsx.relative_to(racine))
print("CSV     :", csv.relative_to(racine))
print("Fichiers sources toujours inchangés :", bundle.verify_unchanged().ok)

Rapport : out/rapport_corroboration.xlsx
CSV     : out/verdicts.csv
Fichiers sources toujours inchangés : True


## Pour aller plus loin

- `corroborai app` — l'interface web : investigation écart par écart, corrections et règles expertes.
- [Guide d'utilisation](../docs/GUIDE_UTILISATEUR.md) · [Architecture](../docs/ARCHITECTURE.md) · [Utilisation de l'IA](../docs/IA.md) · [Règles](../docs/REGLES.md) · [Hypothèses et limites](../docs/HYPOTHESES_ET_LIMITES.md)